# Taller 2 - Punto 3: Fine-tuning de xlm-roberta-large para Analisis de Sentimientos

Este notebook implementa el fine-tuning del modelo xlm-roberta-large de Facebook para clasificacion de sentimientos en tweets en espanol.

## Objetivos:
1. Autenticacion en Hugging Face
2. Carga y preprocesamiento del dataset de Tweets
3. Tokenizacion con xlm-roberta-large
4. Configuracion y entrenamiento del modelo con class weights
5. Evaluacion con metricas de clasificacion (Accuracy, F1, Precision, Recall)
6. Visualizacion del progreso de entrenamiento
7. Subida del modelo a Hugging Face Hub

Este punto requiere GPU para entrenar xlm-roberta-large de forma eficiente.

## 3.1 Configuracion Inicial y Autenticacion en Hugging Face

Reemplaza el token con tu propio token de Hugging Face.
Puedes obtenerlo en: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
import warnings
warnings.filterwarnings('ignore')

# Cargar token desde archivo .env
import os
from pathlib import Path

env_path = Path('.env')
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            if line.startswith('HUGGINGFACE_TOKEN='):
                HF_TOKEN = line.strip().split('=', 1)[1]
                break
else:
    HF_TOKEN = "TU_TOKEN_AQUI"
    print("Advertencia: Archivo .env no encontrado. Usando token por defecto.")

# Autenticacion
try:
    login(HF_TOKEN)
    print("Autenticacion exitosa en Hugging Face")
except Exception as e:
    print(f"Error en autenticacion: {e}")
    print("Por favor, verifica tu token de Hugging Face")

In [ ]:
# Imports necesarios
import os
import re
import json
import pandas as pd
import numpy as np
import torch
from torch import nn
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback
)
import matplotlib.pyplot as plt
import seaborn as sns

# Importar funciones del archivo helpers
from helpers import (
    ensure_directories,
    save_experiment_results,
    load_experiment_results,
    save_training_history,
    load_training_history,
    plot_training_history
)

print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3.2 Configuracion de Directorios y Parametros

In [ ]:
# Configuracion de directorios
MODELS_DIR = "./models/punto3/"
OUTPUT_DIR = "./output/punto3/"
RESULTS_DIR = "./results/"
DATA_DIR = "./input/"

# Crear directorios necesarios
ensure_directories([MODELS_DIR, OUTPUT_DIR, RESULTS_DIR])

# Configuracion del modelo y entrenamiento
MODEL_NAME = "xlm-roberta-large"
MODEL_CHECKPOINT = "FacebookAI/xlm-roberta-large"
MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500

# Configuracion de evaluacion
EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"
METRIC_FOR_BEST_MODEL = "f1"
EARLY_STOPPING_PATIENCE = 3

print("Configuracion establecida:")
print(f"- Modelo: {MODEL_NAME}")
print(f"- Checkpoint: {MODEL_CHECKPOINT}")
print(f"- Max Length: {MAX_LENGTH}")
print(f"- Batch Size: {BATCH_SIZE}")
print(f"- Learning Rate: {LEARNING_RATE}")
print(f"- Epocas: {NUM_EPOCHS}")
print(f"- Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"- Models Directory: {MODELS_DIR}")
print(f"- Output Directory: {OUTPUT_DIR}")
print(f"- Results Directory: {RESULTS_DIR}")

## 3.3 Carga y Preprocesamiento de Datos

Cargamos los datasets de tweets con sentimientos (N=Negativo, P=Positivo).

In [ ]:
def clean_text(text):
    """
    Limpia y normaliza el texto de tweets
    """
    if not isinstance(text, str):
        return ""
    
    # Convertir a minusculas
    text = text.lower()
    
    # Eliminar URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Eliminar menciones de usuario
    text = re.sub(r'@\w+', '', text)
    
    # Eliminar hashtags (mantener el texto)
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # Eliminar caracteres especiales y numeros
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    
    # Eliminar espacios multiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Cargar datasets
print("Cargando datasets...")
df1 = pd.read_csv(f"{DATA_DIR}cr-tass.csv")
df2 = pd.read_csv(f"{DATA_DIR}cr1-tass.csv")

print(f"\nDataset 1 (cr-tass.csv): {len(df1)} muestras")
print(f"Dataset 2 (cr1-tass.csv): {len(df2)} muestras")

# Combinar datasets
df = pd.concat([df1, df2], ignore_index=True)
print(f"\nDataset combinado: {len(df)} muestras")

# Mostrar distribucion de clases original
print("\nDistribucion de clases (original):")
print(df['sentiment'].value_counts())

# Filtrar solo clases N y P
df = df[df['sentiment'].isin(['N', 'P'])].copy()
print(f"\nDespues de filtrar N y P: {len(df)} muestras")

# Limpiar textos
print("\nLimpiando textos...")
df['text_clean'] = df['text'].apply(clean_text)

# Eliminar textos vacios
df = df[df['text_clean'].str.len() > 0].copy()
print(f"Despues de eliminar textos vacios: {len(df)} muestras")

# Convertir etiquetas a numeros: N=0, P=1
label_map = {'N': 0, 'P': 1}
df['label'] = df['sentiment'].map(label_map)

# Mostrar distribucion final
print("\nDistribucion de clases (final):")
print(df['label'].value_counts())
print(f"\nClase 0 (Negativo): {(df['label']==0).sum()} muestras ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"Clase 1 (Positivo): {(df['label']==1).sum()} muestras ({(df['label']==1).sum()/len(df)*100:.1f}%)")

# Mostrar ejemplos
print("\nEjemplos de tweets procesados:")
for i in range(3):
    print(f"\n{i+1}. Sentimiento: {df.iloc[i]['sentiment']}")
    print(f"   Original: {df.iloc[i]['text'][:100]}")
    print(f"   Limpio: {df.iloc[i]['text_clean'][:100]}")

## 3.4 Division de Datos y Tokenizacion

In [ ]:
# Dividir en train y test (80-20)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text_clean'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Dividir train en train y validation (80-20 del train, es decir 64-16-20 total)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

print("Division de datos:")
print(f"- Train: {len(train_texts)} muestras")
print(f"- Validation: {len(val_texts)} muestras")
print(f"- Test: {len(test_texts)} muestras")

# Verificar distribucion de clases
print("\nDistribucion de clases:")
print(f"Train - N: {train_labels.count(0)}, P: {train_labels.count(1)}")
print(f"Val   - N: {val_labels.count(0)}, P: {val_labels.count(1)}")
print(f"Test  - N: {test_labels.count(0)}, P: {test_labels.count(1)}")

# Cargar tokenizer
print(f"\nCargando tokenizer: {MODEL_CHECKPOINT}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Tokenizer cargado exitosamente")

# Funcion de tokenizacion
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

# Crear datasets de HuggingFace
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})
test_dataset = Dataset.from_dict({'text': test_texts, 'label': test_labels})

# Tokenizar datasets
print("\nTokenizando datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Establecer formato
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenizacion completada")

# Mostrar ejemplo tokenizado
print("\nEjemplo de tokenizacion:")
print(f"Texto: {train_texts[0][:100]}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(train_dataset[0]['input_ids'])[:20]}")

## 3.5 Configuracion del Modelo con Class Weights

Configuramos el modelo con pesos de clase para manejar el desbalanceo de datos.

In [ ]:
# Calcular class weights para balancear clases
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"Class weights calculados:")
print(f"  Clase 0 (Negativo): {class_weights[0]:.4f}")
print(f"  Clase 1 (Positivo): {class_weights[1]:.4f}")

# Definir Trainer personalizado con weighted loss
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Aplicar weighted cross entropy loss
        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()
        
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("\nWeightedLossTrainer configurado")

# Cargar modelo
print(f"\nCargando modelo: {MODEL_CHECKPOINT}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    problem_type="single_label_classification"
)

print(f"Modelo cargado exitosamente")
print(f"Numero de parametros: {sum(p.numel() for p in model.parameters()):,}")

## 3.6 Funciones de Metricas y Callbacks

In [ ]:
# Funcion para calcular metricas
def compute_metrics(eval_pred):
    """
    Calcula accuracy, F1, precision y recall
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    precision = precision_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Callback para guardar historial de entrenamiento
class MetricsHistoryCallback(TrainerCallback):
    def __init__(self):
        self.history = {
            'train_loss': [],
            'eval_loss': [],
            'eval_accuracy': [],
            'eval_f1': [],
            'eval_precision': [],
            'eval_recall': [],
            'epoch': []
        }
    
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        """Guardar metricas despues de cada evaluacion"""
        self.history['epoch'].append(state.epoch)
        self.history['eval_loss'].append(metrics.get('eval_loss', None))
        self.history['eval_accuracy'].append(metrics.get('eval_accuracy', None))
        self.history['eval_f1'].append(metrics.get('eval_f1', None))
        self.history['eval_precision'].append(metrics.get('eval_precision', None))
        self.history['eval_recall'].append(metrics.get('eval_recall', None))
    
    def on_log(self, args, state, control, logs, **kwargs):
        """Guardar loss de entrenamiento"""
        if 'loss' in logs:
            self.history['train_loss'].append(logs['loss'])

# Crear callback
metrics_callback = MetricsHistoryCallback()

print("Funciones de metricas y callbacks configurados")
print("\nMetricas a calcular:")
print("  - Accuracy")
print("  - F1 Score (weighted)")
print("  - Precision (weighted)")
print("  - Recall (weighted)")

## 3.7 Configuracion de TrainingArguments y Entrenamiento

In [ ]:
# Configurar argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    logging_dir=f"{OUTPUT_DIR}logs/",
    logging_steps=50,
    evaluation_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=True,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)

print("TrainingArguments configurados:")
print(f"  - Output dir: {OUTPUT_DIR}")
print(f"  - Epochs: {NUM_EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Weight decay: {WEIGHT_DECAY}")
print(f"  - Warmup steps: {WARMUP_STEPS}")
print(f"  - Metric for best model: {METRIC_FOR_BEST_MODEL}")
print(f"  - FP16: {torch.cuda.is_available()}")

# Crear trainer
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        metrics_callback
    ]
)

print("\nTrainer configurado con:")
print(f"  - Weighted loss (class weights)")
print(f"  - Early stopping (patience={EARLY_STOPPING_PATIENCE})")
print(f"  - Metrics history callback")

# Entrenar modelo
print("\n" + "="*80)
print("INICIANDO ENTRENAMIENTO")
print("="*80)

train_result = trainer.train()

print("\n" + "="*80)
print("ENTRENAMIENTO COMPLETADO")
print("="*80)
print(f"\nMetricas de entrenamiento:")
print(f"  - Training loss: {train_result.training_loss:.4f}")
print(f"  - Training runtime: {train_result.metrics['train_runtime']:.2f} segundos")
print(f"  - Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

## 3.8 Evaluacion en Test Set

In [ ]:
# Evaluar en test set
print("="*80)
print("EVALUACION EN TEST SET")
print("="*80)

test_results = trainer.evaluate(test_dataset)

print("\nResultados en Test Set:")
print(f"  - Loss: {test_results['eval_loss']:.4f}")
print(f"  - Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"  - F1 Score: {test_results['eval_f1']:.4f}")
print(f"  - Precision: {test_results['eval_precision']:.4f}")
print(f"  - Recall: {test_results['eval_recall']:.4f}")

# Obtener predicciones para reporte detallado
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# Reporte de clasificacion
print("\n" + "="*80)
print("REPORTE DE CLASIFICACION DETALLADO")
print("="*80)
print("\nClase 0: Negativo (N)")
print("Clase 1: Positivo (P)")
print("\n" + classification_report(test_labels, pred_labels, target_names=['Negativo', 'Positivo']))

# Matriz de confusion
cm = confusion_matrix(test_labels, pred_labels)
print("\n" + "="*80)
print("MATRIZ DE CONFUSION")
print("="*80)
print(f"\n{'':15} {'Pred Negativo':>15} {'Pred Positivo':>15}")
print(f"{'Real Negativo':<15} {cm[0,0]:>15} {cm[0,1]:>15}")
print(f"{'Real Positivo':<15} {cm[1,0]:>15} {cm[1,1]:>15}")

# Visualizar matriz de confusion
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'],
            ax=ax)
ax.set_xlabel('Prediccion')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusion - Test Set')
plt.tight_layout()
confusion_matrix_path = f"{OUTPUT_DIR}confusion_matrix.png"
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches='tight')
print(f"\nMatriz de confusion guardada en: {confusion_matrix_path}")
plt.show()

## 3.9 Visualizacion del Historial de Entrenamiento

In [ ]:
# Guardar historial de entrenamiento
training_history = {
    'history': metrics_callback.history,
    'train_result': {
        'training_loss': float(train_result.training_loss),
        'train_runtime': float(train_result.metrics['train_runtime']),
        'train_samples_per_second': float(train_result.metrics['train_samples_per_second'])
    },
    'test_results': {
        'loss': float(test_results['eval_loss']),
        'accuracy': float(test_results['eval_accuracy']),
        'f1': float(test_results['eval_f1']),
        'precision': float(test_results['eval_precision']),
        'recall': float(test_results['eval_recall'])
    },
    'confusion_matrix': cm.tolist(),
    'configuration': {
        'model': MODEL_NAME,
        'checkpoint': MODEL_CHECKPOINT,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'weight_decay': WEIGHT_DECAY,
        'warmup_steps': WARMUP_STEPS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE
    },
    'dataset_info': {
        'train_size': len(train_texts),
        'val_size': len(val_texts),
        'test_size': len(test_texts),
        'num_classes': 2,
        'class_names': ['Negativo', 'Positivo']
    }
}

save_training_history(training_history, "punto3_training_history", punto=3)
print("Historial de entrenamiento guardado")

# Visualizar historial con helper function
plot_training_history(training_history, punto=3)
print(f"Graficas de entrenamiento guardadas en: {OUTPUT_DIR}")

## 3.10 Guardar Modelo y Subir a Hugging Face Hub

In [ ]:
# Guardar modelo localmente
model_save_path = f"{MODELS_DIR}xlm-roberta-sentiment-final/"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Modelo guardado localmente en: {model_save_path}")

# Configurar para subir a Hugging Face Hub
HF_MODEL_NAME = "tu-usuario/xlm-roberta-sentiment-spanish"

print("\n" + "="*80)
print("SUBIR MODELO A HUGGING FACE HUB")
print("="*80)
print(f"\nPara subir el modelo a Hugging Face Hub:")
print(f"1. Reemplaza 'tu-usuario' con tu usuario de HuggingFace en HF_MODEL_NAME")
print(f"2. Asegurate de estar autenticado (ya lo hiciste al inicio)")
print(f"3. Ejecuta el siguiente codigo:")
print(f"\n# trainer.push_to_hub('{HF_MODEL_NAME}')")
print(f"\nEsto subira el modelo a: https://huggingface.co/{HF_MODEL_NAME}")

# Descomentar para subir el modelo:
# trainer.push_to_hub(HF_MODEL_NAME)
# print(f"Modelo subido exitosamente a: https://huggingface.co/{HF_MODEL_NAME}")

print("\n" + "="*80)
print("NOTA: Comentado por defecto. Descomentar para subir el modelo.")
print("="*80)

## 3.11 Conclusiones y Resumen de Resultados

In [ ]:
# Generar reporte final
print("="*80)
print("RESUMEN Y CONCLUSIONES - PUNTO 3")
print("="*80)

print("\n1. CONFIGURACION DEL EXPERIMENTO:")
print(f"   - Modelo: {MODEL_NAME}")
print(f"   - Checkpoint: {MODEL_CHECKPOINT}")
print(f"   - Tamano de lote: {BATCH_SIZE}")
print(f"   - Tasa de aprendizaje: {LEARNING_RATE}")
print(f"   - Epocas entrenadas: {NUM_EPOCHS}")
print(f"   - Max length: {MAX_LENGTH}")

print("\n2. INFORMACION DEL DATASET:")
print(f"   - Total de muestras: {len(df)}")
print(f"   - Train: {len(train_texts)} muestras")
print(f"   - Validation: {len(val_texts)} muestras")
print(f"   - Test: {len(test_texts)} muestras")
print(f"   - Clases: Negativo (N), Positivo (P)")

print("\n3. TECNICAS APLICADAS:")
print("   - Limpieza de texto (URLs, menciones, caracteres especiales)")
print("   - Tokenizacion con xlm-roberta-large")
print("   - Weighted loss para balancear clases")
print("   - Early stopping con patience=3")
print("   - Evaluacion con metricas multiples")

print("\n4. RESULTADOS EN TEST SET:")
print(f"   - Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"   - F1 Score: {test_results['eval_f1']:.4f}")
print(f"   - Precision: {test_results['eval_precision']:.4f}")
print(f"   - Recall: {test_results['eval_recall']:.4f}")

print("\n5. MATRIZ DE CONFUSION:")
print(f"   - Verdaderos Negativos: {cm[0,0]}")
print(f"   - Falsos Positivos: {cm[0,1]}")
print(f"   - Falsos Negativos: {cm[1,0]}")
print(f"   - Verdaderos Positivos: {cm[1,1]}")

# Calcular tasas
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

print("\n6. TASAS DE CLASIFICACION:")
print(f"   - Tasa de Verdaderos Negativos (Especificidad): {specificity:.4f}")
print(f"   - Tasa de Verdaderos Positivos (Sensibilidad): {sensitivity:.4f}")

print("\n7. OBSERVACIONES PRINCIPALES:")
print("   - El modelo xlm-roberta-large es efectivo para analisis de sentimientos")
print("   - Los class weights ayudan a manejar el desbalanceo de clases")
print("   - Las metricas muestran un buen balance entre precision y recall")
print("   - El modelo converge adecuadamente segun el historial de entrenamiento")

print("\n8. LIMITACIONES Y MEJORAS FUTURAS:")
print("   - Dataset relativamente pequeno (podria mejorarse con mas datos)")
print("   - Solo considera dos clases (N y P)")
print("   - Podria explorarse data augmentation para aumentar el dataset")
print("   - Considerar ensemble con otros modelos pre-entrenados")

print("\n9. ARCHIVOS GENERADOS:")
print(f"   - Modelo guardado: {model_save_path}")
print(f"   - Historial de entrenamiento: {RESULTS_DIR}")
print(f"   - Visualizaciones: {OUTPUT_DIR}")
print(f"   - Matriz de confusion: {confusion_matrix_path}")

print("\n" + "="*80)
print("EXPERIMENTO COMPLETADO EXITOSAMENTE")
print("="*80)

# Guardar reporte final
final_report = {
    "timestamp": datetime.now().isoformat(),
    "experiment_name": "Punto 3 - Fine-tuning xlm-roberta-large",
    "configuration": training_history['configuration'],
    "dataset_info": training_history['dataset_info'],
    "results": {
        "test_metrics": training_history['test_results'],
        "confusion_matrix": training_history['confusion_matrix'],
        "specificity": float(specificity),
        "sensitivity": float(sensitivity)
    },
    "model_info": {
        "save_path": model_save_path,
        "hf_model_name": HF_MODEL_NAME,
        "num_parameters": sum(p.numel() for p in model.parameters())
    },
    "training_info": training_history['train_result'],
    "output_files": {
        "model": model_save_path,
        "visualizations": OUTPUT_DIR,
        "results": RESULTS_DIR
    }
}

save_experiment_results(
    experiment_name="punto3_final_report",
    results=final_report,
    punto=3
)

print(f"\nReporte final guardado en: {RESULTS_DIR}punto3_final_report.json")